In [24]:
from pyspark.sql import SparkSession
import json
import os
from pyspark.sql import functions as sf
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "forge-commerce-user")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "forge-commerce-pass")
S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio:9000")
PREFIX = "products"
RAW_BUCKET = "raw"
CLEANED_BUCKET = "cleaned"
CURATED_BUCKET = "curated"
RAW_PATH = f"s3a://{RAW_BUCKET}/{PREFIX}/"
CLEANED_PATH = f"s3a://{CLEANED_BUCKET}/{PREFIX}/"
CURATED_PATH = f"s3a://{CURATED_BUCKET}/{PREFIX}/"

In [25]:
spark = (
        SparkSession.builder.appName("test_customers")
        .master(os.environ.get("SPARK_MASTER", "spark://spark-master:7077"))
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Delta Lake configurations
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .getOrCreate()
    )

In [26]:
df_cleaned = spark.read.format("delta").load(CURATED_PATH)
df_cleaned.printSchema()
df_cleaned.orderBy("product_id", sf.asc("created_at")).select("product_id", "effective_from", "effective_to", "is_active", "sk_product").show(10, False)

root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- color: string (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- description: string (nullable = true)
 |-- dimensions: string (nullable = true)
 |-- inventory_level: long (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- is_discontinued: boolean (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- margin: double (nullable = true)
 |-- material: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_rating: double (nullable = true)
 |-- product_uuid: string (nullable = true)
 |-- return_policy_days: long (nullable = true)
 |-- review_count: long (nullable = true)
 |-- sku: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- supplier_country: string (nullable = true)
 |-- supplier_name: st

+----------+-------------------+-------------------+---------+----------+
|product_id|effective_from     |effective_to       |is_active|sk_product|
+----------+-------------------+-------------------+---------+----------+
|1         |2025-03-18 00:00:00|2025-05-06 00:00:00|false    |1         |
|1         |2025-05-06 00:00:00|NULL               |true     |2         |
|2         |2024-06-02 00:00:00|2024-10-20 00:00:00|false    |3         |
|2         |2024-10-20 00:00:00|NULL               |true     |4         |
|3         |2024-08-05 00:00:00|2026-03-17 00:00:00|false    |5         |
|3         |2026-03-17 00:00:00|NULL               |true     |6         |
|4         |2025-09-28 00:00:00|2025-10-25 00:00:00|false    |7         |
|4         |2025-10-25 00:00:00|NULL               |true     |8         |
|5         |2024-04-11 00:00:00|2025-06-06 00:00:00|false    |9         |
|5         |2025-06-06 00:00:00|NULL               |true     |10        |
+----------+-------------------+------

In [27]:
# count all rows
print("Number of rows: ", df_cleaned.count())

# count distinct sk_customer
print("Number of distinct sk_product: ", df_cleaned.select("sk_product").distinct().count())

#count distinct customer ids
print("Number of distinct product ids: ", df_cleaned.select("product_id").distinct().count())

Number of rows:  1999


Number of distinct sk_product:  1999


Number of distinct product ids:  1000


In [23]:
spark.stop()